In [1]:
from pyspark.sql import SparkSession
from delta import *

In [2]:
builder = SparkSession.builder \
    .appName("Local_Lakehouse_70GB_GPU") \
    .master("local[2]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.driver.memory", "5g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "1g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "1000") \
    .config("spark.sql.files.maxPartitionBytes", "16777216") \
    .config("spark.sql.caseSensitive", "true")

In [3]:
spark = configure_spark_with_delta_pip(builder).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/11 14:52:11 WARN Utils: Your hostname, LAPTOP-MSL0MUJB, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/11 14:52:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/harsh/wsl_venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/harsh/.ivy2.5.2/cache
The jars for the packages stored in: /home/harsh/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d94831b0-7391-40a6-8bec-0ac323e501a8;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache

In [4]:
post_df = spark.read \
    .format("parquet") \
    .load("/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/readyForSentimentAnalysis/")

In [5]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [6]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon',quiet=True)
from typing import Iterator
import pandas as pd


In [7]:
sia=SentimentIntensityAnalyzer()

In [8]:

# 1. Define a Pandas UDF (Batched for Arrow)
@pandas_udf(FloatType())
def vader_udf_func(iterator: Iterator[pd.Series]) -> Iterator[pd.Series]:
    
    # Initialize VADER *inside* the worker process!
    # This prevents PySpark from trying to serialize it over the network.
    sia = SentimentIntensityAnalyzer()
    
    for text_batch in iterator:
        # Fill nulls with empty strings to prevent crashes
        clean_batch = text_batch.fillna("")
        
        # Apply VADER to the whole batch efficiently
        scores = clean_batch.apply(lambda text: sia.polarity_scores(text)['compound'] if text else 0.0)
        
        yield scores

In [9]:
post_df=post_df.withColumn("vader_sentiment_score",vader_udf_func(col("commit.record.text")))

In [10]:
post_df.select(col("commit.record.text").alias("text"),col("vader_sentiment_score")).filter(col("text").contains("trump")).show(n=100,truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------+
|text                                                                                                                                                                                                                                                                                                                    |vader_sentiment_score|
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------

In [11]:
post_df.write.format("parquet")\
       .mode("append")\
       .save("/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/vaderSentimentTable/")